# 02 DEBATE relevance filter (posts)

Zero-shot filter on the cleaned posts with Political DEBATE (`mlburnham/Political_DEBATE_large_v1.0`, Burnham et al., 2026). -> https://huggingface.co/mlburnham/Political_DEBATE_base_v1.0

Each post is scored on the hypothesis that it describes or asks about personal experiences of joining
a political movement or ideological community. Posts with a score of at least 0.5 are kept (`eligibility = 1`).

Input: the cleaned posts from step 01 (`..._submissions_cleaned.csv`). Run once per subreddit.

## Settings
Results are saved per chunk, so if the run stops it continues from where it was.

In [ ]:
import shutil
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline

# input file and output folder (change these for each subreddit)
INPUT_FILE = Path("path/to/Subreddit_submissions_cleaned.csv")
OUT_DIR    = Path("path/to/output_folder")
TEXT_COL   = "datapoint"

# model + hypothesis definition
MODEL    = "mlburnham/Political_DEBATE_large_v1.0"
TEMPLATE = "This post {}."
LABELS = {
    "entry": "describes or asks about personal experiences of joining or becoming involved "
             "in a political movement or ideological community",
}
THRESHOLD = 0.5
WINDOW_TOKENS, WINDOW_OVERLAP = 380, 80   # window size / overlap (fits in 512 with the hypothesis)
BATCH_SIZE = 32          # windows per batch
CHUNK_SIZE = 10_000      # posts per saved chunk
RECALL_N, SEED = 100, 42 # rejected posts for the recall check

## Model and scoring

In [ ]:
# load the model (apple gpu / cuda / cpu)
device = "mps" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else -1)
pipe = pipeline("zero-shot-classification", model=MODEL, device=device)
tok = pipe.tokenizer
HYPOTHESES = list(LABELS.values())
HYP_TO_LABEL = {h: k for k, h in LABELS.items()}
print(f"Model loaded on {device}")


def windows(text):
    # short posts stay as one window, long ones get split with overlap
    ids = tok(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= WINDOW_TOKENS:
        return [text]
    step = WINDOW_TOKENS - WINDOW_OVERLAP
    return [tok.decode(ids[s:s + WINDOW_TOKENS], skip_special_tokens=True)
            for s in range(0, len(ids) - WINDOW_OVERLAP, step)]


def score_posts(texts):
    # flatten all windows and remember which post each one belongs to
    flat, owner = [], []
    for i, t in enumerate(texts):
        for w in windows(t):
            flat.append(w)
            owner.append(i)
    # score in batches, failed batches stay empty
    scores = [{}] * len(flat)
    for b in range(0, len(flat), BATCH_SIZE):
        try:
            res = pipe(flat[b:b + BATCH_SIZE], HYPOTHESES,
                       hypothesis_template=TEMPLATE, multi_label=True)
            for j, r in enumerate(res if isinstance(res, list) else [res]):
                scores[b + j] = {HYP_TO_LABEL[h]: s for h, s in zip(r["labels"], r["scores"])}
        except Exception as e:
            print(f"  batch failed ({e}) -> its posts get missing scores")
    # max score per post (over its windows)
    ws = pd.DataFrame(scores, columns=list(LABELS)).assign(post=owner)
    return ws.groupby("post")[list(LABELS)].max().reindex(range(len(texts)))

## Run

In [ ]:
df = pd.read_csv(INPUT_FILE, low_memory=False)
texts = df[TEXT_COL].fillna("").astype(str).tolist()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# score chunk by chunk, skip chunks that are already saved
chunk_dir = OUT_DIR / "_chunks"
chunk_dir.mkdir(exist_ok=True)
for start in tqdm(range(0, len(texts), CHUNK_SIZE)):
    f = chunk_dir / f"{start:08d}.csv"
    if not f.exists():
        score_posts(texts[start:start + CHUNK_SIZE]).to_csv(f, index=False)
sc = pd.concat([pd.read_csv(p) for p in sorted(chunk_dir.glob("*.csv"))], ignore_index=True)
assert len(sc) == len(df), "chunk rows do not match input rows"

# add score and labels
failed = sc["entry"].isna()
passed = sc["entry"] >= THRESHOLD
df["score_entry"] = sc["entry"].values
df["eligibility"] = passed.astype(float).where(~failed).values
df["DEBATE_label"] = (passed.map({True: "entry", False: "not_relevant"})
                      .where(~failed, "failed").values)

# save all / eligible / recall sample, then delete the chunks
eligible = df[df["eligibility"] == 1]
rejected = df[df["eligibility"] == 0]
recall = rejected.sample(min(RECALL_N, len(rejected)), random_state=SEED)[
    ["id", TEXT_COL, "score_entry", "DEBATE_label"]].assign(manual_relevant="")
df.to_csv(OUT_DIR / "debate_all.csv", index=False)
eligible.to_csv(OUT_DIR / "debate_eligible.csv", index=False)
recall.to_csv(OUT_DIR / "recall_sample_TO_CODE.csv", index=False)
shutil.rmtree(chunk_dir)

## Summary
Saved as `debate_counts.csv`. The `manual_relevant` column (1/0) in the recall sample is coded by hand.

In [ ]:
# counts: posts before / after the filter, failed posts
counts = pd.Series({"posts": len(df), "eligible": len(eligible), "rejected": len(rejected),
                    "failed": int(failed.sum())})
display(counts)
counts.to_csv(OUT_DIR / "debate_counts.csv", header=["n"])